# 🧊 Pure functions

*Section 15.2 of the notes.*

Beyond its value, we want three more things from a computation:

* its **derivative** (`grad`),
* the same computation on a whole **batch** of inputs (`vmap`),
* a **compiled** version that runs without the interpreter (`jit`).

Each comes from handing the function to an operation that returns a new
function, a **transformation**. A transformation can only do this if the
function is a genuine map from arguments to results: everything it uses
arrives through its arguments, and everything it produces leaves through its
return value. Such a function is called **pure**. 🧊

In Python, purity is up to the programmer. This notebook shows what goes wrong
without it, writes the forward Euler method in two styles, checks whether
purity costs memory, and ends with a transformation that silently gives a
wrong answer on code that is not pure. 😱

## ⚙️ Setup

In [ ]:
import tracemalloc
from functools import reduce

import jax
import numpy as np

## 1. 🔍 Three functions, one pure

The three functions below add a number `b` to a list, or keep a running sum.

* `add_bias` changes the list it is given, **in place**;
* `add_bias_pure` builds a **new** list;
* `accumulate` reads and writes a variable `total` that lives **outside** it.
  The line `global total` tells Python that `total` is that outside variable,
  not a new variable of the function.

🤔 *Guess what each `print` shows before running the cell.*

In [ ]:
def add_bias(values, b):
    for i in range(len(values)):
        values[i] += b
    return values


def add_bias_pure(values, b):
    return [v + b for v in values]


total = 0.0


def accumulate(x):
    global total
    total += x
    return total


xs = [1.0, 2.0]
ys = add_bias(xs, 1.0)
print("in place:  xs =", xs, " ys =", ys)

xs = [1.0, 2.0]
ys = add_bias_pure(xs, 1.0)
print("pure:      xs =", xs, " ys =", ys)

first = accumulate(1.0)
second = accumulate(1.0)
print("global:    first call", first, " second call", second)

💡 **What happened?**

* `add_bias` returned the result **and** changed the caller's list. The second
  effect is invisible at the point of call: reading `add_bias(xs, 1.0)` does
  not tell you that `xs` is gone.
* `add_bias_pure` left `xs` alone. ✅
* `accumulate(1.0)` gave two **different** answers to the same call, because
  its answer depends on `total`. To test it, you must reset `total` first.

This is the same mechanism as the views of Section 13.4: a function that
writes into a slice changes the array of its caller.

## 2. 📏 Forward Euler, two ways

One step of the forward Euler method (Section 2.1) for $\mathrm d u/\mathrm d t = f(u)$ is
$u \mapsto u + \Delta t\, f(u)$. We write it twice:

* as **functions**, which take the state and return the new state;
* as an **object**, which keeps the state in `self.u` and changes it.

The test equation is $f(u) = -u$, whose exact solution is $e^{-t}$.

In [ ]:
def euler_step(u, dt, f):
    return u + dt * f(u)


def euler_run(u0, dt, steps, f):
    u = u0
    for _ in range(steps):
        u = euler_step(u, dt, f)
    return u


class EulerSolver:
    def __init__(self, f, dt):
        self.f = f
        self.dt = dt
        self.u = None

    def start(self, u0):
        self.u = u0

    def step(self):
        self.u = euler_step(self.u, self.dt, self.f)
        return self.u


def decay(u):
    return -u

🤔 *Guess: we call each version twice, in exactly the same way. What prints?*

In [ ]:
first = euler_step(1.0, 0.01, decay)
second = euler_step(1.0, 0.01, decay)
print("function: first call", first, " second call", second)

solver = EulerSolver(decay, 0.01)
solver.start(1.0)
first = solver.step()
second = solver.step()
print("object:   first call", first, " second call", second)

💡 **What happened?** The function gives the same number for the same
arguments, whatever was computed before. The object gives a different number
the second time, because it **remembers where it was**: the answer depends on
the history of calls, which the call itself does not show. To repeat a result,
the object must first be put back into the right state with `start`.

## 3. 🧩 Functions of functions

Because the vector field `f` is an **argument**, the solver works for any
equation without being edited. And because one step is a map from state to
state, we can build that map once and apply it repeatedly.

`make_step(dt, f)` returns a **function**: the map $u\mapsto u+\Delta t f(u)$.
It is written with `lambda u: ...`, which defines a function without a name.

In [ ]:
def make_step(dt, f):
    return lambda u: euler_step(u, dt, f)


step = make_step(0.01, decay)  # a function of the state only
print("step(1.0) =", step(1.0))

`functools.reduce(g, items, start)` runs the loop

```python
u = start
for item in items:
    u = g(u, item)
```

and returns `u`. Our `g` is `lambda u, _: step(u)`: it ignores the item, written
`_`, so `range(100)` only counts the steps. The result is `step` applied 100
times to $u_0 = 1$: the same loop as `euler_run`, written as one expression.

In [ ]:
u_reduce = reduce(lambda u, _: step(u), range(100), 1.0)
u_loop = euler_run(1.0, 0.01, 100, decay)

print("100 steps with reduce:  ", u_reduce)
print("100 steps with euler_run:", u_loop)
print("exact e^(-1):            ", np.exp(-1.0))

💡 After 100 steps of size $0.01$ we are at $t = 1$. Euler gives
$(1 - 0.01)^{100} \approx 0.366$, close to $e^{-1} \approx 0.368$.

A batch of initial states needs no new code: a list comprehension calls the
same function once per initial state. The solver knows nothing about batches.

In [ ]:
u0s = [1.0, 2.0, 3.0]
u_batch = [euler_run(u0, 0.01, 100, decay) for u0 in u0s]
print("initial states:", u0s)
print("after 100 steps:", u_batch)

## 4. 💾 Does purity cost memory?

A pure step returns a **new** state instead of overwriting the old one. Take a
state of $10^6$ numbers, 8 MB, and take 50 steps of $u \mapsto u - \Delta t\,u$.

* `step_new` returns the new state as a new array.
* `step_into` writes the new state into an array `out` that the caller created
  once. `np.multiply(u, c, out=out)` computes $c\,u$ and writes it into `out`.
* `tracemalloc` records the memory that Python allocates.
  `tracemalloc.get_traced_memory()` returns two numbers in bytes: the memory
  held now, and the **largest** amount held at any moment since `start()`.
  We divide by $2^{20}$ bytes to get MiB.

🤔 *Guess: each step of `step_new` creates a new 8 MB array. What is the peak
memory over 50 steps?*

In [ ]:
def step_new(u, dt):
    # The new state, as a new array.
    return u - dt * u


def step_into(u, dt, out):
    # The new state, written into the buffer out that the caller passed in.
    np.multiply(u, 1.0 - dt, out=out)
    return out


u = np.ones(10**6)
tracemalloc.start()
for _ in range(50):
    u = step_new(u, 0.01)
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"new arrays:   peak {peak / 2**20:5.1f} MiB")

u = np.ones(10**6)
out = np.empty(10**6)  # the buffer, created once
tracemalloc.start()
for _ in range(50):
    u = step_into(u, 0.01, out)
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"into buffer:  peak {peak / 2**20:5.1f} MiB")
print("the result is the buffer:", u is out)

💡 **What happened?**

* **Passing** an array to a function copies nothing: Python hands over a
  reference, pure or not.
* What costs memory is **producing** the new value: `u - dt * u` needs two
  temporaries of 8 MB (7.6 MiB), one for the product and one for the
  difference (Section 13.2). The peak is three arrays, 22.9 MiB, because the
  temporaries of one step are released before the next step allocates its own.
* When even that is too much, the caller creates one buffer and passes it in.
  The function is then no longer pure, but it is still decided entirely by its
  arguments. Use it only when a measurement says the allocation matters.

## 5. 😱 Why transformations need pure functions

Now the payoff. `jax.jit` compiles a function. To do that it **traces** it:
it runs the function once with placeholders and records the operations it
sees. Anything the function does **outside** its arguments and return value is
not recorded.

We compile the function `euler_run` and the method `solver.step`, and call
each three times.

* `jit` traces only array arguments. The time step, the number of steps, and
  the vector field are fixed by the `lambda`, so the compiled function
  `run_jit` has one argument, the initial state.
* `float(...)` turns the array that a compiled function returns into a Python
  number.

🤔 *Guess: does the compiled object still advance one step per call?*

In [ ]:
run_jit = jax.jit(lambda u0: euler_run(u0, 0.01, 100, decay))
for call in (1, 2, 3):
    u = float(run_jit(1.0))
    print(f"function, compiled, call {call}: {u:.4f}")
print()

solver = EulerSolver(decay, 0.01)
solver.start(1.0)
for call in (1, 2, 3):
    u = float(solver.step())
    print(f"object, plain,      call {call}: {u:.4f}")
print()

solver.start(1.0)
step_jit = jax.jit(solver.step)
for call in (1, 2, 3):
    u = float(step_jit())
    print(f"object, compiled,   call {call}: {u:.4f}")
print("self.u afterwards:", solver.u)

💡 **What happened?** The compiled function gives the right answer. The
compiled method returns $0.99$ **every time**, and no error is reported. 😱

`solver.step` has no arguments, so there was nothing to trace: during
tracing the arithmetic on `self.u` ran once in plain Python, `self.u` became
$0.99$, and the compiled function is the constant $0.99$. Later calls return
that constant and never touch `self.u` again, which is why it still holds
$0.99$ after three calls.

👉 This is why JAX asks for pure functions: everything the computation needs
must come in as an argument, and everything it produces must go out as a
result. The next notebook applies `grad`, `vmap`, and `jit` to `euler_run`
without changing a line of it.

## 🎯 Summary

| | pure function | in place / global state / object |
|---|---|---|
| same call, same answer | ✅ | ❌ depends on history |
| test | one line | set up, call in order, reset |
| run in parallel | ✅ nothing shared | ⚠️ calls can interfere |
| `grad`, `vmap`, `jit` | ✅ | ❌ wrong or refused, sometimes silently |
| memory | temporaries of one step | can reuse a buffer |

The arrangement that works: a **functional core**, where every computation is
a function of its arguments, inside an **object shell** that carries the
parameters and the configuration (the Flax notebook).

## 🧪 Try it yourself

1. Rewrite `accumulate` as a pure function `accumulate_pure(total, x)` that
   returns the new total. How does the caller keep the running sum now?
2. In Section 4, replace 50 steps by 500. Does the peak of `step_new` change?
3. In Section 5, call `solver.start(5.0)` and then `step_jit()`. What does it
   return, and why?